# **FICORO_GNSS**
### An open-source Python software package for filtering, combining and rotating GNSS velocity fields
### Nicolás Castro-Perdomo, 2024
#### Indiana University, Bloomington, US
---

### Define input and output parameters

In [ ]:
# Import libraries and define paths and parameters
import os
import subprocess
import datetime
import shutil
import sys
import glob
import pandas as pd
import numpy as np
import pygmt
import json
import tempfile
import warnings
import concurrent.futures
import psutil
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt

# Input parameters
input_path = "raw_input"
output_path = "raw_input_column_formatted"
input_file_ext = "raw"
output_file_ext = "vel"
scripts_path = "scripts"
results_path = "results"
igb_nocomb_path = "igb14_no_comb"
igb_nocomb_subpath = "igb14"
log_path = "logs"

lognorm_script_path = os.path.join(scripts_path, "lognorm_filter.py")
coherence_script_path = os.path.join(scripts_path, "coherence_filter.py")
combination_script_path = os.path.join(scripts_path, "combine_vel.py")
plot_maps_filter_path = os.path.join(scripts_path, "plot_maps_filtering.py")
coherence_results_path = os.path.join(results_path, "output_coherence_analysis")
input_files_rot_path = os.path.join(results_path, "input_files_rotation")
ITRF14_vel_path = os.path.join(results_path, "rotation_steps")
lnk_file = "velrot.lnk"
lnk_folder_path = "./rotation_lnk_file"
lnk_file_path = os.path.join(lnk_folder_path, lnk_file)
reference_vel = "serpelloni_2022.vel"
reference_vel_path = os.path.join(input_files_rot_path, reference_vel)
temp_combined_folder_path = os.path.join(results_path, "combined_velocities")
combined_folder_path = os.path.join(temp_combined_folder_path, "manual_filter")
plot_rotated_script_path = os.path.join(scripts_path, "plot_rotated_vels.py")
# Output folder for combined horizontal velocity fields with scaled uncertainties
output_folder_scaled_results = os.path.join(results_path, "combined_velocities_scaled_uncertainties")  

print("------------------------------------------------------------------------------------")
print(f"                     FICORO_GNSS run time: {datetime.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')}               ")
print("------------------------------------------------------------------------------------")
print(" Description: Python tools to filter, rotate and combine GNSS velocity fields       ")
print(" Default parameters:                                                                ")
print(f" - Input path including raw velocity files= {input_path}/                          ")
print(f" - Output path including combined horizontal velocities= {output_folder_scaled_results}/             ")
print(f" - Reference velocity field for alignment to ITRF14= {reference_vel}               ")
print(" - Headers: Lon Lat E.vel N.vel E.adj N.adj E.sig N.sig Corr U.vel U.adj U.sig Sta    ")
print("------------------------------------------------------------------------------------")
print()

---
### Load input velocity fields

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from io_utils import format_raw_velocity_files

format_raw_velocity_files(input_path, output_path, input_file_ext, output_file_ext)

## **Step 1: Filter out stations affected by postseismic transients**

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from velocity_processing import filter_postseismic_stations

# Paths
input_file_path = os.path.join(output_path, "ergintav_2023.vel")
log_file_path   = os.path.join(log_path, "ergintav_2023_removed_postseismic.log")
os.makedirs(log_path, exist_ok=True)

# Epicentres for Izmit, Duzce, and Halkidiki — see Ergintav et al. (2023)
epicentres         = [(29.86, 40.75), (31.16, 40.76), (22, 40)]
distance_threshold = 150  # km

filter_postseismic_stations(
    input_file_path, log_file_path, epicentres, distance_threshold
)

---
## **Steps 2 and 3: Filter stations by velocity uncertainties and spatial coherence**

In [ ]:
# Running GNSS velocity cleaning/filtering scripts
print() 
print("------------------------------------------------------------------------------------")
print("                          Cleaning GNSS velocity fields                             ")
print("------------------------------------------------------------------------------------")
print()                

###########################################################################################
#########################  Execute lognormal filter script ###############################
###########################################################################################

if os.path.exists(lognorm_script_path):
    input_lognorm_path = "./raw_input_column_formatted"
    output_lognorm_path = "./results/output_lognorm_99_filtered"
    excluded_lognorm_path = "./results/sites_excluded_lognorm_99"
    figure_folder_path = "./results/figures"
    
    subprocess.run(["python", lognorm_script_path, input_lognorm_path, output_lognorm_path, excluded_lognorm_path, figure_folder_path])
else:
    print(f"Error: {lognorm_script_path} not found. Terminating")
    exit(1)

print()     

###########################################################################################
#####  Coherence filter with optional user-defined geographic-based stringency levels #####
###########################################################################################

# Enable or disable geographic stringency feature 
geo_strict_enabled = True  # Change to False to disable this feature

# Define geographic regions with variable stringency levels. The default sigma value is 2.
regions = [
    {'min_lon': 117.93, 'max_lon': 123.16, 'min_lat': 21.85, 'max_lat': 25.75, 'sigma': 3}, # Taiwan
    {'min_lon': 34.35, 'max_lon': 36.87, 'min_lat': 29.54, 'max_lat': 36.24, 'sigma': 3}, # Dead Sea Fault area
    {'min_lon': 29, 'max_lon': 34, 'min_lat': 40, 'max_lat': 40.15, 'sigma': 3}, # Izmit and Ismetpasa creeping faults  
    # Add more regions as needed...
]

# Path to the folder containing output data from the lognormal filter
input_folder_path = output_lognorm_path

# Define the special case file to be skipped in the coherence analysis
# Define the pattern for special case files
special_case_file = 'ozarpaci' # Velocities from Ozarpaci et al., 2020, GJI, along Izmit and Ismetpasa faults aren't filtered to preserve afterslip creeping signal

# Check if any file contains the pattern in its name
special_case_found = False
for filename in os.listdir(input_folder_path):
    if special_case_file in filename:
        special_case_found = True
        break

# If no file with the special case pattern is found
if not special_case_found:
    print(f"Error: No file containing '{special_case_file}' found in {input_folder_path}. Terminating.")
    exit(1)


# Check if the coherence filter script exists
if not os.path.exists(coherence_script_path):
    print(f"Error: {coherence_script_path} not found. Terminating")
    exit(1)

# Execute the coherence filter script

if geo_strict_enabled:
    # Create a temporary file to store the regions if the feature is enabled
    with tempfile.NamedTemporaryFile(delete=False, mode='w', suffix='.json') as tmpfile:
        regions_json_path = tmpfile.name
        json.dump(regions, tmpfile)
    
    command = [
        "python", coherence_script_path, input_folder_path,
        '--geo_strict', '--regions_json', regions_json_path, '--special_case_file', special_case_file
    ]
else:
    command = [
        "python", coherence_script_path, input_folder_path,
        '--special_case_file', special_case_file
    ]

# Execute the coherence filter script
if os.path.exists(coherence_script_path):
    subprocess.run(command)
    if geo_strict_enabled:
        # Remove the temporary JSON file after the analysis
        os.remove(regions_json_path)
else:
    print(f"Error: {coherence_script_path} not found. Terminating")
    exit(1)

#### Applying a Lower Bound to Velocity Uncertainties for Certain Input Velocity Fields Reporting Unrealistically Low Values
- Some input velocity fields report unrealistically low velocity uncertainties (e.g., < 0.1 mm/yr), likely due to the use of formal uncertainties derived from fitting a trajectory model to GNSS timeseries, considering only white noise.
- Such low uncertainties are problematic, as they can introduce numerical instabilities when performing weighted least-squares inversions for strain rates.
- To address this, I identified velocity fields reporting formal uncertainties and imposed a lower bound of 0.5 mm/yr on them to ensure stability in the inversion process.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from velocity_processing import apply_uncertainty_lower_bound

# Files with unrealistically low uncertainties that need a lower bound applied (in mm/yr)
file_names = ["bahrouni_2020"]

apply_uncertainty_lower_bound(coherence_results_path, file_names, lower_bound=0.5)

---
#### Plot filtered GNSS velocities and outliers for each input velocity field

In [ ]:
# User flag: set to True to plot, False to skip plotting
do_plots = False  # Change to True to enable plotting

if do_plots:
    print()
    print("------------------------------------------------------------------------------------")
    print("                          Plotting GNSS velocity fields                             ")
    print("------------------------------------------------------------------------------------")
    print()

    # Execute plotting script
    if os.path.exists(plot_maps_filter_path):
        subprocess.run(["python", plot_maps_filter_path])
    else:
        print(f"Error: {plot_maps_filter_path} not found. Terminating")
        exit(1)
else:
    print("Plotting skipped because do_plots is False.")

---
## Step 4: Align input velocity fields to ITRF14 

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from io_utils import convert_csv_to_vel
from itrf14_alignment import align_to_itrf14, collect_igb14_files

pyvelrot_path = os.path.join(os.getcwd(), "scripts", "pyvelrot.py")

# 1. Reformat CSV outputs to velrot-compatible .vel files
convert_csv_to_vel(coherence_results_path, input_files_rot_path)

# 2. Align each .vel file to ITRF14 via pyvelrot
align_to_itrf14(
    input_vel_folder   = input_files_rot_path,
    rotation_folder    = ITRF14_vel_path,
    reference_vel_path = reference_vel_path,
    lnk_file_path      = lnk_file_path,
    pyvelrot_path      = pyvelrot_path,
)

# 3. Collect IGB14 results into the shared igb14_no_comb folder
igb_nocomb_subfolder = os.path.join(results_path, igb_nocomb_path, igb_nocomb_subpath)
collect_igb14_files(ITRF14_vel_path, igb_nocomb_subfolder)
print("Alignment to IGB14 complete.")

---
## Step 5: Rotate velocity fields using Euler poles

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from combination_runner import run_euler_rotations

# Euler pole cartesian components [wx, wy, wz] in DEG/My
euler_poles = {
    "arab": "0.32840 -0.03504 0.40682",   # Arabia — Viltres et al. (2022)
    "sina": "0.24250 -0.04797 0.34939",   # Sinai — Hamiel et al. (2021)
    "anat": "1.008722 0.543127 1.020384", # Anatolia — Ergintav et al. (2023)
    "nubi": "0.02740 -0.1704 0.2037",     # Nubia — Altamimi et al. (2016)
    "eura": "-0.0235 -0.1476 0.2140",     # Eurasia — Altamimi et al. (2016)
    "indi": "0.3205 -0.0014 0.4038",      # India — Altamimi et al. (2016)
    "amur": "-0.04800 -0.10714 0.27414",  # Amur — Castro-Perdomo et al. (2025)
    "yang": "-0.0578 -0.1272 0.2933",     # Yangtze — Vardic et al. (2022)
    "aege": "0.050898 0.147426 0.158303", # W. Aegean — Ergintav et al. (2023)
    "tbet": "0.12983 -0.71360 -0.02336",  # C. Tibet — Castro-Perdomo et al. (2025)
}

pycvframe_path = os.path.join(os.getcwd(), "scripts", "pycvframe.py")
igb14_folder   = os.path.join(results_path, igb_nocomb_path, igb_nocomb_subpath)

run_euler_rotations(euler_poles, igb14_folder, igb_nocomb_path,
                    results_path, pycvframe_path)

---
## Steps 6 and 7: Combination and filtering by velocity magnitude and azimuthal direction

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from combination_runner import run_velocity_combination

# Allowed values: "auto", "parallel", "sequential"
execution_mode = "auto"

frames = list(euler_poles.keys()) + ["igb14"]

run_velocity_combination(
    frames                  = frames,
    combination_script_path = combination_script_path,
    igb_nocomb_path         = igb_nocomb_path,
    results_path            = results_path,
    output_folder           = temp_combined_folder_path,
    mode                    = execution_mode,
)

### **Number of independent velocity estimates at each GNSS station**

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from plot_gps_velocity_fields import plot_number_of_estimates

plot_number_of_estimates(
    "results/combined_velocities/statistics/site_statistics.csv"
)

---
## Step 8: Manual filtering of outliers

- Here I implement an approach that involves filtering GNSS stations given a list of coordinates and radii.
- The primary objective is removing stations affected by volcanic transients and outliers not removed in the previous filtering steps.


In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from velocity_processing import apply_manual_filter

temp_combined_folder_path = "./results/combined_velocities/"
combined_folder_path      = "./results/combined_velocities/manual_filter/"
filter_criteria_path      = "./manual_filter/filter_criteria.csv"

print("--------------------------------------------------------------")
print("            Filtering GNSS velocity fields manually            ")
print("--------------------------------------------------------------")
print()

apply_manual_filter(temp_combined_folder_path, combined_folder_path, filter_criteria_path)

---
### Plot combined velocity field in different reference frames

In [ ]:
# User flag: set to True to enable plotting
do_plots = False  # Change to True to enable plotting

if do_plots:
    # Plotting rotated velocities before cleaning
    print("---------------------------------------------------")
    print("      Plotting final combined velocity fields      ")
    print("---------------------------------------------------")
    print()

    subprocess.run(["python", plot_rotated_script_path, combined_folder_path])

    print()
    print("--------------------------------------------------------------")
    print("Plotting combined velocity fields in different reference frames completed")
    print()
else:
    print("Plotting skipped because do_plots is False.")

#### Final results: Clean combined horizontal velocity fields in different reference frames

In [ ]:
"""Plotting horizontal velocity fields in different reference frames"""

sys.path.append(os.path.abspath("./scripts"))
from plot_gps_velocity_fields import plot_gps_velocity_fields

input_folder = "./results/combined_velocities/manual_filter/"

# Plot all reference frames:
#plot_gps_velocity_fields(input_folder)

# To plot a single plate (e.g. Eurasia-fixed only), use:
plot_gps_velocity_fields(input_folder, plate_name="eura")


#### **Combine vertical GNSS velocity fields**

- Here, I simply compute the median vertical velocity for each station considering a ridius of 1.2 km

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from io_utils import prepare_vertical_input_files

# Input velocity fields that contain reliable vertical velocities
input_files = [
    "./results/input_files_rotation/serpelloni_2022.vel",
    "./results/input_files_rotation/viltres_2022.vel",
    # ergintav_2023.vel excluded — unreliable vertical uncertainties
    "./results/input_files_rotation/euref_all.vel",
    "./results/input_files_rotation/floyd_2023.vel",
    "./results/input_files_rotation/euref_ch8.vel",
    "./results/input_files_rotation/sokhadze_2018.vel",
    "./results/input_files_rotation/castro_2021.vel",
    "./results/input_files_rotation/briole_2021.vel",
    "./results/input_files_rotation/liang_2013.vel",
    "./results/input_files_rotation/pinaValdes_2022.vel",
    "./results/input_files_rotation/jolivet_2023.vel",
]

# liang_2013 reports 0.00 for stations without verticals; convert to NaN
modify_files       = ["liang_2013"]
destination_folder = "./results/input_verticals/"
raw_input_folder   = "./raw_input/levelling_verticals/"

prepare_vertical_input_files(
    input_files, destination_folder, modify_files, raw_input_folder
)

print(f"\nStaged vertical input files in {destination_folder}:")
for f in sorted(os.listdir(destination_folder)):
    print(f"  {f}")

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
import combine_verticals as cv

destination_folder = "./results/input_verticals/"
output_combined_verticals = "./results/combined_velocities/manual_filter"
log_file_path = "./results/combined_velocities/manual_filter/debug_combined_vertical_velocity_field.log"

output_file_path, _ = cv.combine_verticals(
    input_folder=destination_folder,
    output_folder=output_combined_verticals,
    log_file_path=log_file_path,
)

print(f"Combined vertical velocity field saved to: {output_file_path}")
print(f"Debug log saved to: {log_file_path}")


In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from plot_gps_velocity_fields import plot_vertical_velocity_histogram
from velocity_processing import filter_vertical_velocities_by_sigma

combined_vertical_file = (
    "./results/combined_velocities/manual_filter/"
    "combined_vertical_velocity_field.vel"
)
n_sigma = 2  # rejection threshold (number of standard deviations)

# Plot histogram with mean ± n_sigma lines
plot_vertical_velocity_histogram(combined_vertical_file, n_sigma=n_sigma)

# Remove outliers and save
filtered_file_path = (
    "./results/combined_velocities/manual_filter/"
    "combined_vertical_velocity_field_mag_filtered.vel"
)
filter_vertical_velocities_by_sigma(combined_vertical_file, filtered_file_path, n_sigma)

In [ ]:
# Add the scripts folder to the Python path
sys.path.append(os.path.abspath('./scripts'))

from uncertainty_filter_verticals import UncertaintyFilterVerticals

# Define the file path and output locations
input_file = './results/combined_velocities/manual_filter/combined_vertical_velocity_field_mag_filtered.vel'
output_folder = './results/combined_velocities/manual_filter/'
figures_path = './results/figures/'

# Create an instance of the UncertaintyFilterVerticals class
filter_verticals = UncertaintyFilterVerticals(input_file, output_folder, figures_path)

# Read the vertical velocities
filter_verticals.read_vertical_velocities()

# Plot the uncertainty distribution
filter_verticals.plot_uncertainty_distribution()

# Filter the uncertainties and save the filtered velocities
filter_verticals.filter_uncertainties()

In [ ]:
# Here I count number of stations in the combined vertical velocity field within longitude and latitude ranges: 
# [5, 60] and [-20, 125], respectively. 

# Load the combined vertical velocity field
combined_vertical_file = './results/combined_velocities/manual_filter/filtered_vertical_velocity_field.vel'
col_names = ['Lon', 'Lat', 'U.vel', 'U.sig', 'Stat']
combined_data = pd.read_csv(combined_vertical_file, sep=r'\s+', header=None, names=col_names)

# Count the number of stations within the specified longitude and latitude ranges (Alpine-Himalayan Belt region)
lon_mask = (combined_data['Lon'] >= 5) & (combined_data['Lon'] <= 60)
lat_mask = (combined_data['Lat'] >= -20) & (combined_data['Lat'] <= 125)
stations_count = len(combined_data[lon_mask & lat_mask])

# Print the result
print(f"Number vertical velocities along the Alpine-Himalayan Belt region: {stations_count}")

In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
from plot_gps_velocity_fields import plot_vertical_velocity_fields

plot_vertical_velocity_fields([combined_vertical_file])

## Steps 9 and 10: Scaling of horizontal GNSS velocity uncertainties and final filtering
- Here I follow the approach described in Piña-Valdez et al., (2022) to scale the distribution of velocity uncertainties in the combined velocity field, taking Wang & Barbot (2023) as the reference/target log-normal distribution of velocity uncertainties.
- Finally, I apply another filtering step based on the lognormal distribution of velocity uncertainties, removing sites with uncertainties exceeding the 99th percentile of the fitted log-normal distribution.

In [ ]:
# Import all functions from the module

# Add the scripts folder to the Python path
sys.path.append(os.path.abspath('./scripts'))

import uncertainty_scaling_combined as uncertainty_scaling

# Set paths and run the harmonisation process
input_folder = './results/combined_velocities/manual_filter/' # Folder containing the combined velocity fields
reference_filename = './results/output_coherence_analysis/wang_barbot_2023.csv' # Reference file for uncertainty scaling
output_folder_scaled_results = './results/combined_velocities_scaled_uncertainties/'  # Output folder for results with scaled uncertainties

# Run the harmonisation/scaling process
uncertainty_scaling.harmonise_uncertainties(input_folder, reference_filename, output_folder_scaled_results)


### Plotting final velocities after uncertainty scaling and filtering:

In [ ]:
# Plotting rotated velocities before cleaning
print("---------------------------------------------------")
print("      Plotting final combined velocity fields      ")
print("---------------------------------------------------")
print()

subprocess.run(["python", plot_rotated_script_path, output_folder_scaled_results])

print()
print("--------------------------------------------------------------")
print("Plotting combined velocity fields in different reference frames completed")
print()